# Fraud detection – regularised logistic regression
**Final model: L2-regularised logistic regression** on log-transformed and flag features.
LightGBM and TabPFN are run only as challengers, to show that more complex models give no significant gain.

**Why logistic regression**
- In earlier runs it scored about the same as the complex models (OOF PR-AUC 0.224 vs TabPFN 0.232 and the blend 0.225), while fold-to-fold noise is ±0.05.
- Every prediction is a sum of feature effects, and each coefficient reads as an odds multiplier (see the odds-ratio table).
- Its assumptions can be stated and checked (see the assumptions section).

**Validation vs. submission format**
- PR-AUC / ROC-AUC compare the true 0/1 labels with **continuous scores**, never with thresholded 0/1 predictions.
- F1 / Recall compare the true labels with 0/1 predictions made by thresholding those scores.
- `submission.csv` holds a **fraud score in [0,1]**: its ranking gives PR-AUC, and it is shifted so the chosen threshold sits at 0.5.

In [1]:
# Set to True to also run TabPFN as a challenger (needs a TABPFN_TOKEN secret and a GPU or a long CPU run)
RUN_TABPFN = False

%pip install -q lightgbm
if RUN_TABPFN:
    %pip install -q tabpfn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys
import numpy as np
import pandas as pd

# Read from the repo's data/ folder if it exists; otherwise (e.g. Colab) open an upload button
def load_csv(name, prompt):
    for path in [f"../data/{name}", f"data/{name}", name]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    print(prompt)
    uploaded = files.upload()
    return pd.read_csv(list(uploaded.keys())[0])

df = load_csv("Track 2 Training Dataset.csv", "Upload the TRAINING csv")
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (20000, 12)


,id,transaction_amount,transaction_hour,merchant_category,country,transaction_channel,transactions_last_24h,spend_last_24h,account_age,new_device,transactions_last_1h,fraud
0,FR016993,55.03,16,retail,SG,ecommerce,27.0,1791.34,1192.0,0.0,5.0,0
1,FR002433,20.75,1,gaming,SG,mobile_app,3.0,217.96,3073.0,0.0,0.0,0
2,FR009253,21.51,9,entertainment,SG,mobile_app,3.0,62.71,985.0,0.0,0.0,0
3,FR004881,813.98,16,utilities,GB,mobile_app,2.0,110.48,653.0,0.0,1.0,0
4,FR014830,13.54,20,food_delivery,SG,card_present,4.0,114.98,79.0,0.0,0.0,0


In [3]:
if RUN_TABPFN:
    # SECURITY: never paste the token into the notebook. In Colab, add it under the key icon (Secrets) as TABPFN_TOKEN.
    from google.colab import userdata
    os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
    os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

In [4]:
import numpy as np

target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]

# Adds per-row features (uses only that row's own values, so no label or cross-row leakage)
def engineer(d):
    d = d.copy()
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# --- DROP UNNECESSARY ID COLUMNS --- (id is a random label; checked: it carries no fraud signal)
X = engineer(df.drop(columns=["id", target_column]))
y = df[target_column].values
print(f"Features being used ({X.shape[1]}): {list(X.columns)}")

# Full numeric set (used by the original logistic spec and by the tree / TabPFN challengers)
FULL_NUM = [c for c in X.columns if c not in CATS]

# Simplified set for logistic regression: drop near-duplicate columns that make coefficients unstable and unreadable
DROP_FOR_LOGIT = [
    "transaction_amount", "spend_last_24h", "account_age",  # log versions kept: test amounts are much larger, so avoid extrapolating raw values
    "transaction_hour",   # a straight line in hour 0-23 is meaningless; hour_sin/hour_cos + night carry the daily cycle
    "risky_merchant",     # exactly the sum of three merchant_category one-hot columns
    "n_flags",            # (almost) exactly the sum of other flags already in the model
    "amt_to_avg_ticket",  # ~ log_amt - log_spend + log(count): VIF ~26, got a misleading negative coefficient
    "young",              # overlaps log_age (sign flipped to "lower risk" next to it)
    "transactions_last_1h", "share_1h",  # overlap burst / transactions_last_24h
    "hour_sin", "hour_cos",              # night already captures the time-of-day risk
]
# Checked with 5x3 repeated CV: dropping the last six changes OOF PR-AUC by +0.003 (SE 0.002), i.e. no loss.
LOGIT_NUM = [c for c in FULL_NUM if c not in DROP_FOR_LOGIT]
print(f"Logistic features ({len(LOGIT_NUM)} numeric + {len(CATS)} categorical): {LOGIT_NUM + CATS}")

Features being used (24): ['transaction_amount', 'transaction_hour', 'merchant_category', 'country', 'transaction_channel', 'transactions_last_24h', 'spend_last_24h', 'account_age', 'new_device', 'transactions_last_1h', 'log_amt', 'log_spend', 'log_age', 'hour_sin', 'hour_cos', 'night', 'burst', 'high24', 'young', 'amt_to_avg_ticket', 'share_1h', 'risky_merchant', 'night_x_device', 'n_flags']
Logistic features (9 numeric + 3 categorical): ['transactions_last_24h', 'new_device', 'log_amt', 'log_spend', 'log_age', 'night', 'burst', 'high24', 'night_x_device', 'merchant_category', 'country', 'transaction_channel']


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb

# Logistic regression; imputation/scaling/encoding are fitted inside the pipeline, so only on training rows.
# Missing indicators are off in the simplified spec: missingness is unrelated to fraud in train, and ~3x more common in test.
def make_logit(num_cols, add_indicator=False):
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=add_indicator), StandardScaler()), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATS),
    ])
    return make_pipeline(pre, LogisticRegression(C=0.3, max_iter=3000))

# Challenger: small, heavily regularised LightGBM (handles NaN and pandas categories natively)
def make_lgb():
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.02, num_leaves=7, min_child_samples=60,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=5, verbose=-1)

MODELS = {
    "logistic_simple": lambda: make_logit(LOGIT_NUM),                           # candidate final model
    "logistic_original": lambda: make_logit(FULL_NUM, add_indicator=True),     # spec from the previous notebook
    "lightgbm": make_lgb,
}
if RUN_TABPFN:
    from tabpfn import TabPFNClassifier
    MODELS["tabpfn"] = lambda: TabPFNClassifier(ignore_pretraining_limits=True)

# Probability of the fraud class (looked up in classes_, not assumed to be column 1)
def p_fraud(model, X_):
    return model.predict_proba(X_)[:, list(model.classes_).index(1)]

In [6]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score

# 5-fold CV repeated twice: every row is scored once per repeat, so each score uses all 353 frauds
N_SPLITS, N_REPEATS = 5, 2
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=0)
oof = {name: np.zeros((N_REPEATS, len(y))) for name in MODELS}   # out-of-fold scores, one row per repeat
fold_ap = {name: [] for name in MODELS}

for i, (tr, te) in enumerate(rskf.split(X, y)):
    r = i // N_SPLITS
    for name, make in MODELS.items():
        model = make().fit(X.iloc[tr], y[tr])
        oof[name][r, te] = p_fraud(model, X.iloc[te])
        fold_ap[name].append(average_precision_score(y[te], oof[name][r, te]))
    print(f"repeat {r} fold {i % N_SPLITS} done")

# PR-AUC / ROC-AUC compare true 0/1 labels with continuous scores (no thresholding)
results = pd.DataFrame({
    name: {
        "PR-AUC (OOF)": np.mean([average_precision_score(y, oof[name][r]) for r in range(N_REPEATS)]),
        "PR-AUC fold std": np.std(fold_ap[name]),
        "ROC-AUC (OOF)": np.mean([roc_auc_score(y, oof[name][r]) for r in range(N_REPEATS)]),
    } for name in MODELS
}).T.round(4)
results

repeat 0 fold 0 done


repeat 0 fold 1 done


repeat 0 fold 2 done


repeat 0 fold 3 done


repeat 0 fold 4 done


...

repeat 1 fold 1 done


repeat 1 fold 2 done


repeat 1 fold 3 done


repeat 1 fold 4 done


,PR-AUC (OOF),PR-AUC fold std,ROC-AUC (OOF)
logistic_simple,0.2319,0.0517,0.7805
logistic_original,0.2298,0.0488,0.7800
lightgbm,0.2159,0.0462,0.7716


### Simplified vs. original logistic spec
The two specs are compared on the **same folds** (paired). We keep the simpler spec unless the original beats it by more than 2 standard errors of the paired difference.

In [7]:
diff = np.array(fold_ap["logistic_original"]) - np.array(fold_ap["logistic_simple"])
se = diff.std(ddof=1) / np.sqrt(len(diff))
print(f"original - simple: mean fold PR-AUC diff = {diff.mean():+.4f}  (SE {se:.4f})")

FINAL = "logistic_simple" if diff.mean() < 2 * se else "logistic_original"
FINAL_NUM = LOGIT_NUM if FINAL == "logistic_simple" else FULL_NUM
print(f"Final model: {FINAL}")

original - simple: mean fold PR-AUC diff = -0.0018  (SE 0.0043)
Final model: logistic_simple


In [8]:
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score

# Choose the share of rows to flag by OOF F1, averaged over repeats. The share is tuned on these same OOF rows, so this F1 is slightly optimistic.
shares = np.arange(0.005, 0.0605, 0.0025)
def f1_at_share(p, s):
    return f1_score(y, (p >= np.quantile(p, 1 - s)).astype(int))
f1s = [np.mean([f1_at_share(oof[FINAL][r], s) for r in range(N_REPEATS)]) for s in shares]
best_share = shares[int(np.argmax(f1s))]

# Detailed report on repeat 0
p_oof = oof[FINAL][0]
threshold = np.quantile(p_oof, 1 - best_share)
y_pred = (p_oof >= threshold).astype(int)   # 0/1 labels, used only for F1 / Recall

print(f"Flagging top {best_share:.2%} of transactions ({y_pred.sum()} alerts), probability threshold = {threshold:.4f}")
print(classification_report(y, y_pred, digits=3))

Flagging top 1.25% of transactions (250 alerts), probability threshold = 0.1411
              precision    recall  f1-score   support

           0      0.987     0.992     0.989     19647
           1      0.348     0.246     0.289       353

    accuracy                          0.979     20000
   macro avg      0.667     0.619     0.639     20000
weighted avg      0.975     0.979     0.977     20000



In [9]:
# Submission score: unweighted logistic probabilities at a 1.8% base rate rarely exceed 0.5, so a 0.5 cut-off
# would flag almost nothing. This strictly increasing map keeps the ranking (so PR-AUC is unchanged)
# and puts our threshold at 0.5: flagged rows -> [0.51, 1], others -> [0, 0.49].
def to_submission_score(score, thr):
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

# Score the OOF predictions exactly as the organisers will score submission.csv
oof_sub = to_submission_score(p_oof, threshold)
assert np.isclose(average_precision_score(y, oof_sub), average_precision_score(y, p_oof)), "ranking changed"
pred_05 = (oof_sub >= 0.5).astype(int)
assert (pred_05 == y_pred).all(), "0.5 cut-off does not reproduce the chosen threshold"

print(f"Submission-format OOF check: PR-AUC = {average_precision_score(y, oof_sub):.4f} (on scores), "
      f"F1@0.5 = {f1_score(y, pred_05):.3f}, Recall@0.5 = {recall_score(y, pred_05):.3f}, "
      f"Precision@0.5 = {precision_score(y, pred_05):.3f}")

Submission-format OOF check: PR-AUC = 0.2259 (on scores), F1@0.5 = 0.289, Recall@0.5 = 0.246, Precision@0.5 = 0.348


## Model assumptions
| Assumption | How we handle / check it |
|---|---|
| **Log-odds are linear in the features** | Skewed amounts/spend/age enter as `log1p`. Known threshold effects (burst ≥4 in 1h, ≥11 in 24h, account ≤30 days, night) enter as 0/1 flags. Time of day enters as the night flag. |
| **Observations are independent** | Cannot be checked: there is no customer id or timestamp. Stated as a limitation; CV is stratified random. |
| **No (near-)perfect multicollinearity** | Duplicate/overlapping columns removed (raw vs. log, `n_flags`, `risky_merchant`, `amt_to_avg_ticket`, ...). Variance inflation factors (VIF) checked below. L2 regularisation (C=0.3) stabilises any remaining correlation. |
| **No perfect separation** | L2 regularisation keeps coefficients finite. |
| **Train and test come from the same distribution** | Violated (test has larger amounts, more new devices, more missing values). Log features and median imputation limit the damage, and the flag share is a stated risk. |

In [10]:
# Variance inflation factor of each numeric feature = diagonal of the inverse correlation matrix. VIF > 10 signals strong collinearity.
Z = SimpleImputer(strategy="median").fit_transform(X[FINAL_NUM])
vif = pd.Series(np.diag(np.linalg.inv(np.corrcoef(Z, rowvar=False))), index=FINAL_NUM, name="VIF")
vif.sort_values(ascending=False).round(2)

transactions_last_24h    2.57
high24                   2.22
log_spend                1.90
burst                    1.68
log_amt                  1.52
night_x_device           1.38
new_device               1.31
night                    1.09
log_age                  1.01
Name: VIF, dtype: float64

## Final model and odds-ratio table
Refit on all 20,000 training rows. Each row of the table shows how the odds of fraud change:
- **0/1 flags:** odds multiplier when the flag goes from 0 to 1.
- **Other numeric features:** odds multiplier for a +1 standard-deviation increase (SD shown in original units).
- **Categories:** odds multiplier vs. the most common level of that column.

Multipliers above 1 raise fraud risk; below 1 lower it. Effects are *holding all other features fixed*, and are shrunk towards 1 by regularisation.

In [11]:
final_model = MODELS[FINAL]().fit(X, y)
names = final_model[:-1].get_feature_names_out()
coef = pd.Series(final_model[-1].coef_[0], index=names)
sds = final_model[0].named_transformers_["num"][-1].scale_   # SDs of the (imputed) numeric inputs, same order as num__ names

rows = []
num_names = [n for n in names if n.startswith("num__")]
for n, sd in zip(num_names, sds):
    feat = n[len("num__"):]
    binary = feat in X.columns and X[feat].dropna().isin([0, 1]).all()
    log_or = coef[n] / sd if binary else coef[n]   # undo scaling for 0/1 flags
    rows.append({"feature": feat, "effect of": "0 → 1" if binary else f"+1 SD ({sd:.3g})", "odds ratio": np.exp(log_or)})
for c in CATS:
    base = X[c].value_counts().idxmax()
    for lvl in final_model[0].named_transformers_["cat"].categories_[CATS.index(c)]:
        if lvl == base:
            continue
        rows.append({"feature": f"{c} = {lvl}", "effect of": f"vs {base}",
                     "odds ratio": np.exp(coef[f"cat__{c}_{lvl}"] - coef[f"cat__{c}_{base}"])})

odds = pd.DataFrame(rows)
odds["direction"] = np.where(odds["odds ratio"] > 1, "↑ risk", "↓ risk")
odds = odds.reindex(odds["odds ratio"].apply(lambda v: abs(np.log(v))).sort_values(ascending=False).index)
pd.set_option("display.max_rows", 100)
odds.round(3).reset_index(drop=True)

,feature,effect of,odds ratio,direction
0,night_x_device,0 → 1,3.177,↑ risk
1,new_device,0 → 1,2.835,↑ risk
2,transaction_channel = Unknown,vs card_present,2.316,↑ risk
3,merchant_category = cash_transfer,vs grocery,2.299,↑ risk
4,burst,0 → 1,2.174,↑ risk
5,merchant_category = luxury,vs grocery,1.981,↑ risk
6,country = ID,vs SG,1.905,↑ risk
7,merchant_category = food_delivery,vs grocery,1.786,↑ risk
8,country = PH,vs SG,1.754,↑ risk
9,merchant_category = electronics,vs grocery,1.741,↑ risk


## Predict the test set and write `submission.csv`
Score the test file with the final model and save `id,prediction`.
`prediction` is a fraud score in [0,1] (not 0/1): its ranking gives PR-AUC, and scores ≥ 0.5 are our fraud flags for F1/Recall.

In [12]:
test_df = load_csv("Track 2 Testing Dataset.csv", "Upload the TEST csv (the one without the fraud column)")
print("Test Shape:", test_df.shape)
assert "fraud" not in test_df.columns, "This looks like the training file - upload the test file instead"

# Same feature steps as training (id kept aside, never used as a feature)
X_test_final = engineer(test_df.drop(columns=["id"]))

# Make test categories use exactly the same category list as training
for c in CATS:
    X_test_final[c] = pd.Categorical(X_test_final[c].astype(str), categories=X[c].cat.categories)
X_test_final = X_test_final[X.columns]

Test Shape: (12000, 11)


In [13]:
test_p = p_fraud(final_model, X_test_final)   # final_model was refit on all 20,000 training rows above

# Flag the same share of test rows as in CV (best_share). Note: test prevalence is unknown and may be higher (see shift notes).
test_threshold = np.quantile(test_p, 1 - best_share)
test_prob = to_submission_score(test_p, test_threshold)

submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": test_prob,   # continuous score in [0,1]; >= 0.5 means flagged as fraud
})

# Sanity checks before writing
assert len(submission) == len(test_df) and submission["id"].is_unique
assert submission["prediction"].notna().all() and submission["prediction"].between(0, 1).all()
assert submission["prediction"].nunique() > 2, "prediction looks binary - PR-AUC needs continuous scores"
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Rows: {len(submission)}, flagged (>= 0.5): {(test_prob >= 0.5).sum()} ({(test_prob >= 0.5).mean():.2%})")

         id  prediction
0  FR021005    0.005291
1  FR030078    0.026028
2  FR020717    0.010715
3  FR026657    0.010538
4  FR023775    0.005081
Rows: 12000, flagged (>= 0.5): 150 (1.25%)


In [14]:
# Download submission.csv when running in Colab
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("submission.csv")